# 02 — Downloading and Local Data

This notebook covers downloading stellar atmosphere spectra from remote sources and managing locally installed catalogs.

**Key methods:** `SED.fetch()`, `SED.local()`, `catalog.write()`

In [2]:
from sed_tools.api import SED

## 1. Downloading a Catalog

`SED.fetch()` downloads spectra from a remote source. By default it tries the NJM mirror first (pre-processed, fastest), then falls back to SVO, MSG, or MAST.

The call returns an `SED` instance whose `.cat` attribute holds the downloaded spectra as a `Catalog` object.

In [3]:
# Basic download — full Kurucz 2003 catalog
sed = SED.fetch('Kurucz2003all')

# Force a specific source
# sed = SED.fetch('Kurucz2003all', source='svo')

Discovering available models from SVO...
Discovering available models from MSG grids...
Discovering available models from MAST (BOSZ 2024)...
[fetch] Trying Kurucz2003all from svo...
  Fetching metadata for Kurucz2003all...
    (cache) 8092 spectra
[fetch] Downloading Kurucz2003all from svo...
    Lookup table saved: /home/njm/SED_Tools/data/stellar_models/Kurucz2003all/lookup_table.csv
  Successfully downloaded 8092/8092 spectra
[fetch] Downloaded 8092 spectra
[clean] Catalog already standardized (all 809 samples)
  [clean] 3000/8092 (37.1%)  elapsed 1s  ETA 2s   

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[fetch] Cleaned: 8092 total, 0 fixed


### Filtered Downloads

You can restrict the download by stellar parameter ranges and parallelise with multiple workers. This is the recommended approach for large catalogs.

In [5]:
# Download only the solar-neighbourhood parameter space
sed = SED.fetch(
 'Kurucz2003all',
 teff_min=4000,
 teff_max=8000,
 logg_min=3.0,
 logg_max=5.0,
 metallicity_min=-1.0,
 metallicity_max=0.5,
 workers=8,
)

Discovering available models from SVO...
Discovering available models from MSG grids...
Discovering available models from MAST (BOSZ 2024)...
[fetch] Trying Kurucz2003all from svo...
  Fetching metadata for Kurucz2003all...
    (cache) 8092 spectra
[fetch] Downloading Kurucz2003all from svo...
    Lookup table saved: /home/njm/SED_Tools/data/stellar_models/Kurucz2003all/lookup_table.csv
  Successfully downloaded 8092/8092 spectra
[fetch] Downloaded 8092 spectra
[clean] Catalog already standardized (all 809 samples)
  [clean] 2997/8092 (37.0%)  elapsed 1s  ETA 2s   

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[fetch] Cleaned: 8092 total, 0 fixed
[fetch] Filtered to 850 spectra


## 2. Saving to Disk

`catalog.write()` generates the full set of MESA-compatible files:
- `lookup_table.csv` — parameter index
- `flux_cube.bin` — binary flux cube consumed by MESA's colors module
- `<name>.h5` — HDF5 bundle of all spectra

All spectra are standardised to wavelength in Å and flux in erg/cm²/s/Å before writing.

In [6]:
# Save to the default data directory
output_path = sed.cat.write()
print(f"Written to: {output_path}")

# Save to a custom path


[write] Wrote 850 spectra to /home/njm/SED_Tools/data/stellar_models/Kurucz2003all
On-collision strategy: all-warn  (source: built-in default)
Lookup table: 850 entries, columns: file_name, teff, logg, metallicity
No extra axes detected — no collisions possible.
Wavelength grid: 1199 points (147.2–1600000.0 Å)
RAM: 22.7 GiB available, using up to 17.0 GiB
Grid: 17 Teff × 5 logg × 5 meta
  Allocating (17, 5, 5, 1199) (0.00 GiB, in RAM)
  spectra 850/850 (100.0%)  skipped 0  ETA 0s   
  Done — 850/850 loaded, 0 skipped.
Writing /home/njm/SED_Tools/data/stellar_models/Kurucz2003all/flux_cube.bin (0.00 GiB, 1 chunk(s))...
  Done (0.0s)

Flux cube: /home/njm/SED_Tools/data/stellar_models/Kurucz2003all/flux_cube.bin
  Teff : 4000 – 8000 K
  logg : 3.00 – 5.00
  [M/H]: -1.00 – 0.50
[write] Built flux_cube.bin
[H5 bundle] Wrote /home/njm/SED_Tools/data/stellar_models/Kurucz2003all/Kurucz2003all.h5
[write] Built Kurucz2003all.h5
Written to: /home/njm/SED_Tools/data/stellar_models/Kurucz2003all


## 3. Loading a Local Catalog

If a catalog is already installed, `SED.local()` loads it instantly from disk without any network access.

In [7]:
try:
    sed = SED.local('Kurucz2003all')
    print("Loaded Kurucz2003all")
except Exception as e:
    print(f"Not found locally: {e}")

Loaded Kurucz2003all


## 4. Inspecting a Local Catalog

Once loaded, you can inspect the grid axes, parameter ranges, and the underlying data files.

In [8]:
if 'sed' in locals():
    ranges = sed.parameter_ranges()
    print("Parameter ranges:")
    for param, (lo, hi) in ranges.items():
        print(f"  {param:12s}: {lo} → {hi}")

    print(f"\nTotal spectra in catalog: {len(sed.cat)}")
    print(f"Unique Teff values: {sed.cat.teff_grid}")
    print(f"Unique logg values: {sed.cat.logg_grid}")

Parameter ranges:
  teff        : 4000.0 → 8000.0
  logg        : 3.0 → 5.0
  metallicity : -1.0 → 0.5

Total spectra in catalog: 850
Unique Teff values: [4000. 4250. 4500. 4750. 5000. 5250. 5500. 5750. 6000. 6250. 6500. 6750.
 7000. 7250. 7500. 7750. 8000.]
Unique logg values: [3.  3.5 4.  4.5 5. ]


## 5. Listing All Installed Models

`SED.query(include_remote=False)` shows everything that is ready to use locally.

In [9]:
local_catalogs = SED.query(include_remote=False)
print(f"Installed models ({len(local_catalogs)}):")
for c in local_catalogs:
    teff = f"{c.teff_range[0]:.0f}–{c.teff_range[1]:.0f} K" if c.teff_range else "unknown"
    print(f"  {c.name:30s}  Teff: {teff}  N: {c.n_spectra}")

Installed models (14):
  Husfeld                         Teff: 35000–80000 K  N: 448
  Kurucz                          Teff: 3500–50000 K  N: 3808
  Kurucz2003                      Teff: 3500–50000 K  N: 3808
  Kurucz2003all                   Teff: 4000–8000 K  N: 850
  Kurucz2003all__alpha_0.4__lh_1.25__vtur_2  Teff: 3500–50000 K  N: 4284
  Kurucz2003all__alpha_00         Teff: 3500–50000 K  N: 3808
  Kurucz2003all__alpha_04         Teff: 3500–50000 K  N: 4284
  Kurucz2003all__alpha_0__lh_1.25__vtur_2  Teff: 3500–50000 K  N: 3808
  Kurucz2003alp                   Teff: 3500–50000 K  N: 4284
  NextGen2                        Teff: 900–10000 K  N: 2712
  bbody                           Teff: 10–200000 K  N: 588
  hres                            Teff: 3000–55000 K  N: 1650
  morley12                        Teff: 400–1300 K  N: 182
  tmap2                           Teff: 20000–150000 K  N: 1215
